In [ ]:
# ── CELL 1: GPU CHECK + MEMORY OPTIMIZATION ────────────────────────────
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # Prevent fragmentation
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # Better error messages

import torch

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
torch.backends.cudnn.benchmark = True

print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU:     {p.name}")
    print(f"Memory:  {p.total_memory/1e9:.1f} GB")
    torch.cuda.empty_cache()

DEVICE = torch.device("cuda:0")
print(f"Device:  {DEVICE}")

In [ ]:
# ── CELL 2: INSTALL TIMM ──────────────────────────────────────────────
import subprocess
result = subprocess.run(
    ["pip", "install", "-q", "timm"],
    capture_output=True, text=True)
print("timm installed")

import timm
print(f"timm version: {timm.__version__}")

In [ ]:
# ── CELL 3: IMPORTS ──────────────────────────────────────────────────────
import os
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import random

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import timm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("✅ All imports done")

In [ ]:

IMAGE_SIZE    = 384        
BATCH_SIZE    = 16   
NUM_EPOCHS    = 130       
NUM_CLASSES   = 8
WARMUP_EPOCHS = 10       

BACKBONE_LR   = 1e-5    
HEAD_LR       = 1e-4    

BACKBONE_MODEL = 'convnext_base' 

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

IMAGES_DIR = '/kaggle/input/datasets/rohandahal123/isic-2019-skin-lesion/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
CSV_PATH   = '/kaggle/input/datasets/rohandahal123/isic-2019-skin-lesion/ISIC_2019_Training_GroundTruth.csv'

CHECKPOINT_DIR  = '/kaggle/working/checkpoints_convnext'
BEST_MODEL_PATH = '/kaggle/working/convnext_base_90pct.pth'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Images dir      : {os.path.exists(IMAGES_DIR)}")
print(f"CSV file        : {os.path.exists(CSV_PATH)}")
print(f"Image size      : {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"Batch size      : {BATCH_SIZE}")
print(f"Model backbone  : {BACKBONE_MODEL}")
print(f"Backbone LR     : {BACKBONE_LR}")
print(f"Head LR         : {HEAD_LR}")
print(f"Warmup          : {WARMUP_EPOCHS} epochs")
print(f"Total epochs    : {NUM_EPOCHS}")
print(f"Expected acc    : 88-92% (single model) → 90-94% (with TTA+ensemble)")

In [ ]:
# ── CELL 5: CLASS DEFINITIONS ─────────────────────────────────────────
DISEASE_CLASSES = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']

CLASS_NAME_MAP = {
    'MEL':  'Melanoma',
    'NV':   'Melanocytic Nevi',
    'BCC':  'Basal Cell Carcinoma',
    'AK':   'Actinic Keratosis',
    'BKL':  'Benign Keratosis',
    'DF':   'Dermatofibroma',
    'VASC': 'Vascular Lesion',
    'SCC':  'Squamous Cell Carcinoma'
}
FULL_CLASS_NAMES = [CLASS_NAME_MAP[c] for c in DISEASE_CLASSES]

print("Classes:")
for i, c in enumerate(DISEASE_CLASSES):
    print(f"  {i}: {CLASS_NAME_MAP[c]}")

In [ ]:
# ── CELL 6: LOAD DATA ────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f"CSV: {df.shape}")

label_cols  = [c for c in df.columns if c in DISEASE_CLASSES]
df['label'] = df[label_cols].values.argmax(axis=1)

df['image_path'] = df['image'].apply(
    lambda x: os.path.join(IMAGES_DIR, x + '.jpg'))

df = df[df['image_path'].apply(os.path.exists)].copy()
df = df.reset_index(drop=True)
print(f"Valid images: {len(df)}")

print("\nClass distribution:")
for i, col in enumerate(DISEASE_CLASSES):
    count = (df['label'] == i).sum()
    bar   = '█' * (count // 400)
    print(f"  {CLASS_NAME_MAP[col]:<30} {count:>5}  {bar}")

In [ ]:
# ── CELL 7: CLASS WEIGHTS + TRAIN/VAL/TEST SPLIT ───────────────────────
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=df['label'].values)
class_weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)

print("Class weights:")
for i, col in enumerate(DISEASE_CLASSES):
    print(f"  {CLASS_NAME_MAP[col]:<30} {class_weights[i]:.4f}")

train_df, temp_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df['label'])
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['label'])

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"\nTrain: {len(train_df)}")
print(f"Val:   {len(val_df)}")
print(f"Test:  {len(test_df)}")

In [ ]:
# ── CELL 8: FOCAL LOSS (NUMERICALLY STABLE) ──────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0,
                 reduction='mean', label_smoothing=0.1):
        super().__init__()
        self.alpha           = alpha
        self.gamma           = gamma
        self.reduction       = reduction
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        n_cls = inputs.size(1)

        smoothed = torch.full_like(
            inputs, self.label_smoothing / (n_cls - 1))
        smoothed.scatter_(
            1, targets.unsqueeze(1), 1.0 - self.label_smoothing)

        log_p = F.log_softmax(inputs, dim=1)

        p_t     = torch.exp(log_p).gather(
            1, targets.unsqueeze(1)).squeeze(1)
        focal_w = (1.0 - p_t.detach()) ** self.gamma

        ce_loss = -(smoothed * log_p).sum(dim=1)
        loss    = focal_w * ce_loss

        if self.alpha is not None:
            loss = loss * self.alpha[targets]

        return loss.mean() if self.reduction == 'mean' else loss


criterion = FocalLoss(
    alpha=class_weights_tensor,
    gamma=2.0,
    label_smoothing=0.1
)
print("✅ FocalLoss ready")

with torch.no_grad():
    dummy_logits  = torch.randn(16, NUM_CLASSES).to(DEVICE)
    dummy_targets = torch.randint(0, NUM_CLASSES, (16,)).to(DEVICE)
    dummy_loss    = criterion(dummy_logits, dummy_targets)
print(f"   Sanity check loss: {dummy_loss.item():.4f}  (expect ~2.0–3.5)")

In [ ]:
# ── CELL 9: MIXUP HELPERS ────────────────────────────────────────────────
def mixup_data(x, y, alpha=0.2):
    lam        = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size = x.size(0)
    index      = torch.randperm(batch_size, device=x.device)
    mixed_x    = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


print("✅ MixUp functions defined")

In [ ]:
# ── CELL 10: DATASET + AUGMENTATION (ENHANCED) ───────────────────────────
try:
    from torchvision.transforms import RandAugment
except ImportError:
    from torch.transforms import RandAugment

class ISICDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = int(row['label'])
        try:
            image = Image.open(row['image_path']).convert('RGB')
        except Exception:
            image = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), (128, 128, 128))
        if self.transform:
            image = self.transform(image)
        return image, label


train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 60, IMAGE_SIZE + 60)),
    transforms.RandomCrop(IMAGE_SIZE),

    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomApply(
        [transforms.RandomRotation(degrees=90)], p=0.8),

    RandAugment(num_ops=3, magnitude=9),
    
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.25,
        hue=0.1),

    transforms.RandomGrayscale(p=0.08),
    
    transforms.RandomAffine(
        degrees=0,
        translate=(0.10, 0.10),
        scale=(0.80, 1.20),
        shear=10),

    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),

    transforms.RandomErasing(p=0.35, scale=(0.02, 0.15), ratio=(0.3, 3.3)),
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

sample_weights = torch.DoubleTensor([
    class_weights[int(l)] for l in train_df['label'].values])
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True)

train_dataset = ISICDataset(train_df, train_transform)
val_dataset   = ISICDataset(val_df,   val_test_transform)
test_dataset  = ISICDataset(test_df,  val_test_transform)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    sampler=sampler, num_workers=4,
    pin_memory=True, persistent_workers=True, drop_last=True)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=4,
    pin_memory=True, persistent_workers=True)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=4,
    pin_memory=True, persistent_workers=True)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")
print("✅ Enhanced augmentation pipeline ready")

In [ ]:
# ── CELL 11: MODEL — CONVNEXT-BASE WITH GEM POOLING ────────────────────
class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p   = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.adaptive_avg_pool2d(
            x.clamp(min=self.eps).pow(self.p),
            output_size=1
        ).pow(1.0 / self.p)


def build_model():
    backbone = timm.create_model(
        BACKBONE_MODEL,
        pretrained=True,
        num_classes=0,
        global_pool=''
    )
    
    # Get correct feature dimension based on backbone
    if 'convnext_base' in BACKBONE_MODEL:
        BACKBONE_FEATURES = 1024  # ConvNeXt-Base output
    elif 'b5' in BACKBONE_MODEL:
        BACKBONE_FEATURES = 2048
    elif 'b6' in BACKBONE_MODEL:
        BACKBONE_FEATURES = 2304
    else:
        # Fallback: infer from dummy forward pass
        with torch.no_grad():
            dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
            out = backbone(dummy)
            BACKBONE_FEATURES = out.shape[1]

    head = nn.Sequential(
        GeM(p=3),
        nn.Flatten(),
        nn.BatchNorm1d(BACKBONE_FEATURES, momentum=0.01),
        nn.Linear(BACKBONE_FEATURES, 512),
        nn.SiLU(),
        nn.Dropout(p=0.5),
        nn.Linear(512, NUM_CLASSES)
    )

    model = nn.Sequential(backbone, head)
    return model.to(DEVICE)


torch.cuda.empty_cache()
model = build_model()

# ── MULTI-GPU SUPPORT: 2x T4 with DataParallel ──────────────────────────
if torch.cuda.device_count() > 1:
    print(f"\n🚀 Multi-GPU detected: {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)
    model = model.to(DEVICE)
    print(f"   Model wrapped with DataParallel")

with torch.no_grad():
    dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
    out   = model(dummy)

print(f"Output shape    : {out.shape}")
print(f"GPU memory      : {torch.cuda.memory_allocated()/1e9:.2f} GB")
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,}")
print(f"✅ ConvNeXt-Base model ready for 90% target (Multi-GPU)")

In [ ]:
# ── CELL 12: OPTIMIZER + SCHEDULER (CONVNEXT TUNED) ────────────────────
backbone_params = []
head_params     = []

for name, param in model.named_parameters():
    if name.startswith('0.'):
        backbone_params.append(param)
    else:
        head_params.append(param)

print(f"Backbone params : {sum(p.numel() for p in backbone_params):,}")
print(f"Head params     : {sum(p.numel() for p in head_params):,}")

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': BACKBONE_LR},
    {'params': head_params,     'lr': HEAD_LR},
], weight_decay=5e-3)

warmup_scheduler = LinearLR(
    optimizer,
    start_factor=0.02,
    end_factor=1.0,
    total_iters=WARMUP_EPOCHS)

cosine_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS - WARMUP_EPOCHS,
    eta_min=1e-8)

scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[WARMUP_EPOCHS])

scaler = GradScaler("cuda")

print(f"\nOptimizer ready")
print(f"  Backbone LR : {BACKBONE_LR}")
print(f"  Head LR     : {HEAD_LR}")
print(f"  Warmup      : {WARMUP_EPOCHS} epochs")
print(f"  Total epochs: {NUM_EPOCHS}")

In [ ]:
# ── CELL 13: TRAIN + VALIDATE FUNCTIONS ──────────────────────────────────
USE_MIXUP   = True
MIXUP_ALPHA = 0.15
MIXUP_START = 16


def train_one_epoch(model, loader, criterion, optimizer, scaler, epoch):
    model.train()
    torch.cuda.empty_cache()

    running_loss = 0.0
    correct      = 0
    total        = 0

    use_mixup = USE_MIXUP and epoch >= MIXUP_START
    if epoch == MIXUP_START:
        print(f"  → MixUp enabled from epoch {epoch} (alpha={MIXUP_ALPHA})")

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda"):
            if use_mixup:
                images, y_a, y_b, lam = mixup_data(images, labels, MIXUP_ALPHA)
                outputs = model(images)
                loss    = mixup_criterion(criterion, outputs, y_a, y_b, lam)
            else:
                outputs = model(images)
                loss    = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, predicted  = outputs.max(1)
        total   += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        if batch_idx % 100 == 0:
            mem = torch.cuda.memory_allocated() / 1e9
            print(f"  [{batch_idx:>4}/{len(loader)}]  "
                  f"Loss: {loss.item():.4f}  "
                  f"Acc: {100.*correct/total:.2f}%  "
                  f"Mem: {mem:.1f}GB")

    return running_loss / len(loader), 100. * correct / total


def validate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with autocast("cuda"):
                outputs = model(images)
                loss    = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted  = outputs.max(1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return running_loss / len(loader), 100. * correct / total


print("✅ train_one_epoch + validate defined")
print(f"   MixUp starts at epoch {MIXUP_START}, alpha={MIXUP_ALPHA}")

In [ ]:
# ── CELL 14: TRAINING LOOP (130 EPOCHS - NO INTERRUPTION) ──────────────────
# Training will complete all 130 epochs regardless of validation accuracy plateaus
best_val_acc     = 0.0
start_epoch      = 1

train_losses     = []
val_losses       = []
train_accuracies = []
val_accuracies   = []

# Resume from checkpoint if one exists
ckpt_files = sorted([
    f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pth')])

if ckpt_files:
    latest = os.path.join(CHECKPOINT_DIR, ckpt_files[-1])
    print(f"Resuming: {latest}")
    ckpt = torch.load(latest, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch      = ckpt['epoch'] + 1
    best_val_acc     = ckpt['best_val_acc']
    train_losses     = ckpt.get('train_losses',     [])
    val_losses       = ckpt.get('val_losses',       [])
    train_accuracies = ckpt.get('train_accuracies', [])
    val_accuracies   = ckpt.get('val_accuracies',   [])
    print(f"Epoch: {start_epoch}, Best so far: {best_val_acc:.2f}%")
else:
    print("Starting fresh with ConvNeXt-Base.")

print(f"\nTraining {start_epoch} → {NUM_EPOCHS} (all epochs, no early stopping)")
print("=" * 60)

for epoch in range(start_epoch, NUM_EPOCHS + 1):

    lrs = [pg['lr'] for pg in optimizer.param_groups]
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}  "
          f"LR: backbone={lrs[0]:.2e}  head={lrs[1]:.2e}")
    print("-" * 50)

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, epoch)

    val_loss, val_acc = validate(model, val_loader, criterion)

    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"\nTrain — Loss: {train_loss:.4f}  Acc: {train_acc:.2f}%")
    print(f"Val   — Loss: {val_loss:.4f}  Acc: {val_acc:.2f}%")

    # Save checkpoint every 10 epochs
    if epoch % 10 == 0:
        ckpt_path = os.path.join(CHECKPOINT_DIR, f'ckpt_epoch_{epoch:03d}.pth')
        torch.save({
            'epoch':                epoch,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict':    scaler.state_dict(),
            'best_val_acc':         best_val_acc,
            'train_losses':         train_losses,
            'val_losses':           val_losses,
            'train_accuracies':     train_accuracies,
            'val_accuracies':       val_accuracies,
        }, ckpt_path)
        print(f"💾 ckpt_epoch_{epoch:03d}.pth")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"✅ New best! Val: {val_acc:.2f}%")

print("\n" + "=" * 60)
print(f"✅ Training complete after all {NUM_EPOCHS} epochs. Best val acc: {best_val_acc:.2f}%")

In [ ]:
# ── CELL 15: TEST-TIME AUGMENTATION (TTA) ────────────────────────────────
def predict_with_tta(model, loader, n_augments=15):
    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            batch_probs = torch.zeros(
                images.size(0), NUM_CLASSES).to(DEVICE)

            for aug_idx in range(n_augments):
                aug = images.clone()
                
                if torch.rand(1) > 0.5:
                    aug = TF.hflip(aug)
                if torch.rand(1) > 0.5:
                    aug = TF.vflip(aug)
                
                if torch.rand(1) > 0.7:
                    aug = TF.rotate(aug, angle=90 * torch.randint(0, 4, (1,)).item())
                
                if torch.rand(1) > 0.8:
                    zoom_factor = torch.rand(1).item() * 0.2 + 0.9
                    h, w = aug.shape[-2:]
                    crop_h, crop_w = int(h / zoom_factor), int(w / zoom_factor)
                    if crop_h > 0 and crop_w > 0:
                        top = torch.randint(0, h - crop_h + 1, (1,)).item() if h > crop_h else 0
                        left = torch.randint(0, w - crop_w + 1, (1,)).item() if w > crop_w else 0
                        aug = TF.crop(aug, top, left, crop_h, crop_w)
                        aug = TF.resize(aug, [h, w])
                
                aug = aug.to(DEVICE)
                with autocast("cuda"):
                    out = model(aug)
                batch_probs += torch.softmax(out, dim=1)

            batch_probs /= n_augments
            preds = batch_probs.argmax(dim=1).cpu()
            all_preds.extend(preds.numpy())
            all_labels.extend(labels.numpy())

    return np.array(all_preds), np.array(all_labels)


# Load best checkpoint and evaluate
model.load_state_dict(
    torch.load(BEST_MODEL_PATH, map_location=DEVICE))

print("Running TTA on validation set (15 augments)...")
tta_preds, tta_labels = predict_with_tta(model, val_loader, n_augments=15)
tta_acc = (tta_preds == tta_labels).mean() * 100

print(f"\nVal accuracy WITHOUT TTA : {best_val_acc:.2f}%")
print(f"Val accuracy WITH TTA    : {tta_acc:.2f}%")
print(f"TTA boost                : +{tta_acc - best_val_acc:.2f}%")

print("\nClassification Report (Val + TTA):")
print(classification_report(
    tta_labels, tta_preds, target_names=FULL_CLASS_NAMES))

print("\nRunning TTA on test set (15 augments)...")
test_preds, test_labels = predict_with_tta(model, test_loader, n_augments=15)
test_acc = (test_preds == test_labels).mean() * 100
print(f"\nTest accuracy WITH TTA: {test_acc:.2f}%")
print("\nClassification Report (Test + TTA):")
print(classification_report(
    test_labels, test_preds, target_names=FULL_CLASS_NAMES))

In [ ]:
# ── CELL 16: CONFUSION MATRIX + TRAINING CURVES (ENHANCED VISUALIZATION) ───
plt.close('all')

old_image_path = '/kaggle/working/convnext_results.png'
if os.path.exists(old_image_path):
    os.remove(old_image_path)
    print(f"🗑️  Deleted old image: {old_image_path}")

fig, axes = plt.subplots(1, 3, figsize=(24, 7))
fig.suptitle('ConvNeXt-Base Training Results', fontsize=20, fontweight='bold', y=1.00)

epochs_range = range(1, len(train_losses) + 1)

# ── Loss plot ──
axes[0].plot(epochs_range, train_losses, label='Train Loss', linewidth=2.5, marker='o', markersize=3)
axes[0].plot(epochs_range, val_losses, label='Val Loss', linewidth=2.5, marker='s', markersize=3)
axes[0].set_title('Loss Progression', fontsize=14, fontweight='bold', pad=10)
axes[0].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Loss', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=11, loc='upper right')
axes[0].grid(True, alpha=0.4, linestyle='--')
axes[0].set_facecolor('#f8f9fa')

# ── Accuracy plot ──
axes[1].plot(epochs_range, train_accuracies, label='Train Acc', linewidth=2.5, marker='o', markersize=3, color='#2ecc71')
axes[1].plot(epochs_range, val_accuracies, label='Val Acc', linewidth=2.5, marker='s', markersize=3, color='#e74c3c')
axes[1].axhline(y=85, color='gray', linestyle='--', linewidth=1.5, alpha=0.7, label='85% Target')
axes[1].axhline(y=90, color='gold', linestyle='--', linewidth=1.5, alpha=0.9, label='90% Target')
axes[1].set_title('Accuracy Progression', fontsize=14, fontweight='bold', pad=10)
axes[1].set_xlabel('Epoch', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
axes[1].set_ylim([0, 105])
axes[1].legend(fontsize=10, loc='lower right')
axes[1].grid(True, alpha=0.4, linestyle='--')
axes[1].set_facecolor('#f8f9fa')

# ── Confusion matrix ──
cm = confusion_matrix(test_labels, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

sns.heatmap(
    cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[2],
    xticklabels=DISEASE_CLASSES, yticklabels=DISEASE_CLASSES,
    cbar_kws={'label': 'Accuracy'}, linewidths=0.5, linecolor='gray',
    vmin=0, vmax=1, annot_kws={'fontsize': 10, 'fontweight': 'bold'})
axes[2].set_title('Confusion Matrix (Test + TTA)', fontsize=14, fontweight='bold', pad=10)
axes[2].set_ylabel('True Label', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
axes[2].tick_params(axis='x', labelsize=10)
axes[2].tick_params(axis='y', labelsize=10)

plt.tight_layout()
plt.savefig('/kaggle/working/convnext_results.png',
            dpi=300, bbox_inches='tight', facecolor='white')
print("✅ Saved NEW convnext_results.png (300 DPI, high quality)")

# Display the figure
plt.show()

# Print final statistics
print("\n" + "="*60)
print("📊 FINAL RESULTS")
print("="*60)
test_acc_pct = (test_preds == test_labels).mean() * 100
print(f"Best Val Accuracy:  {best_val_acc:.2f}%")
print(f"Test Accuracy:      {test_acc_pct:.2f}%")
print(f"Total Epochs:       {len(train_losses)}")
print("="*60)